# 01 — Tokenization Cache

**Prerequisite**: `00_csi_pipeline.ipynb` must have been run — `datasets/UNIFIED.jsonl` must exist.

### What this notebook does
1. **Load** UNIFIED.jsonl + config.yaml
2. **Tokenize** all 4,085 records with GraphCodeBERT tokenizer
3. **Inspect** token length distribution (important for `max_length` tuning)
4. **Cache** input_ids + attention_mask to disk as `.pt` files
5. **Verify** cache integrity + print stats
6. **Define** `VulnerabilityDataset` + `get_dataloaders()` — reused by `02_training.ipynb`

### Platform support
| Platform | Notes |
|---|---|
| Apple Silicon (M1/M2/M3) | PyTorch MPS or CPU — tokenization is CPU-bound anyway |
| Google Colab / CUDA | Standard PyTorch — tokenization runs same speed |

> Cache takes ~2–5 min on first run. Subsequent runs load in seconds.

## 0 — Install Dependencies

In [14]:
import subprocess, sys, platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in __import__("os").environ

print(f"Platform      : {platform.system()} {platform.machine()}")
print(f"Apple Silicon : {IS_APPLE_SILICON}")
print(f"Colab         : {IS_COLAB}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.40",
        "peft>=0.10",
        "torch",
        "pyyaml",
        "tqdm",
    ],
    check=False,
)
print("Done.")

Platform      : Darwin arm64
Apple Silicon : True
Colab         : False
Done.


## 1 — Paths, Config & Helpers

In [15]:
import os, json, platform, sys
from pathlib import Path
import yaml

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"

try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("Colab — Drive mounted")
except ImportError:
    BASE_DIR = Path(os.path.dirname(os.path.abspath("__file__")))
    if not (BASE_DIR / "datasets").exists():
        BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f"Local — BASE_DIR: {BASE_DIR}")

# Load config
cfg_path = BASE_DIR / "config.yaml"
assert cfg_path.exists(), f"Missing config.yaml at {cfg_path}"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

UNIFIED_PATH = BASE_DIR / cfg["unified_jsonl"]
TOKEN_CACHE_DIR = BASE_DIR / cfg["token_cache_dir"]
TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = cfg["model_name"]
MAX_LENGTH = cfg["max_length"]
BATCH_SIZE = cfg["tokenizer_batch_size"]

assert (
    UNIFIED_PATH.exists()
), f"Missing: {UNIFIED_PATH} — run 00_csi_pipeline.ipynb first"


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


print(f"UNIFIED_PATH    : {UNIFIED_PATH}")
print(f"TOKEN_CACHE_DIR : {TOKEN_CACHE_DIR}")
print(f"MODEL_NAME      : {MODEL_NAME}")
print(f"MAX_LENGTH      : {MAX_LENGTH}")

Local — BASE_DIR: /Users/anas/Projects/code-security-identifier
UNIFIED_PATH    : /Users/anas/Projects/code-security-identifier/datasets/UNIFIED.jsonl
TOKEN_CACHE_DIR : /Users/anas/Projects/code-security-identifier/datasets/token_cache
MODEL_NAME      : microsoft/graphcodebert-base
MAX_LENGTH      : 512


## 2 — Load UNIFIED Dataset

In [16]:
records = read_jsonl(UNIFIED_PATH)
print(f"Loaded {len(records):,} records")

# CWE label mapping (must match 00_csi_pipeline.ipynb)
CWE_8_CLASSES = [
    "CWE-077",
    "CWE-601",
    "CWE-022",
    "CWE-094",
    "CWE-089",
    "CWE-352",
    "CWE-079",
    "unknown",
]
CWE_TO_INDEX = {cwe: i for i, cwe in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE = {i: cwe for cwe, i in CWE_TO_INDEX.items()}


def map_cwe_to_label(cwe_id: str) -> int:
    return CWE_TO_INDEX.get(cwe_id, CWE_TO_INDEX["unknown"])


# Quick sanity
train_records = [r for r in records if r["split_origin"] == "train"]
val_records = [r for r in records if r["split_origin"] == "val"]
print(f"  train : {len(train_records):,}")
print(f"  val   : {len(val_records):,}")

# CWE label distribution
from collections import Counter

lbl_dist = Counter(map_cwe_to_label(r["cwe_id"]) for r in records)
print()
print("CWE label distribution:")
for lbl in sorted(lbl_dist):
    print(f"  {lbl} ({INDEX_TO_CWE[lbl]:<10}) : {lbl_dist[lbl]:,}")

Loaded 4,085 records
  train : 3,677
  val   : 408

CWE label distribution:
  0 (CWE-077   ) : 194
  1 (CWE-601   ) : 209
  2 (CWE-022   ) : 232
  3 (CWE-094   ) : 122
  4 (CWE-089   ) : 589
  5 (CWE-352   ) : 181
  6 (CWE-079   ) : 404
  7 (unknown   ) : 2,154


## 3 — Init Tokenizer + Inspect Token Lengths

In [17]:
from transformers import AutoTokenizer
import statistics

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_ctx = getattr(tokenizer, "model_max_length", 512)
if model_ctx is None or model_ctx > 100000:
    model_ctx = 512

if MAX_LENGTH > model_ctx:
    print(
        f"Warning: MAX_LENGTH={MAX_LENGTH} exceeds model limit ({model_ctx}). Clamping to {model_ctx}."
    )
    MAX_LENGTH = model_ctx

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Model max context length: {model_ctx}")

# Sample token length distribution (analysis pass: no padding, no truncation)
print("\nSampling token lengths (analysis pass: no padding, no truncation)...")
sample_lengths = []
for r in records:
    code = " ".join(r["lines"])
    # verbose=False prevents tokenizer max-length warning in analysis-only pass
    enc = tokenizer(
        code,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        verbose=False,
    )
    sample_lengths.append(len(enc["input_ids"]))

pcts = [50, 75, 90, 95, 99, 100]
sorted_lengths = sorted(sample_lengths)
n = len(sorted_lengths)

print(f"\nToken length stats ({len(sample_lengths):,} functions):")
print(f"  min    : {min(sample_lengths)}")
print(f"  mean   : {statistics.mean(sample_lengths):.1f}")
print(f"  median : {statistics.median(sample_lengths)}")
for p in pcts:
    idx = min(int(p / 100 * n), n - 1)
    print(f"  p{p:<3}   : {sorted_lengths[idx]}")

truncated = sum(1 for l in sample_lengths if l > MAX_LENGTH)
trunc_pct = 100 * truncated / n
print(
    f"\nTruncated at max_length={MAX_LENGTH}: {truncated:,} / {n:,}  ({trunc_pct:.1f}%)"
)
if trunc_pct > 20:
    print(
        f"  -> {MODEL_NAME} is capped at {model_ctx} tokens. "
        "Use chunking/windowing or line-level localization to reduce information loss."
    )
else:
    print("  -> Truncation is within the target range (<20%).")

print("\nEstimated truncation by candidate max_length values:")
for candidate in [256, 384, 512]:
    t = sum(1 for l in sample_lengths if l > candidate)
    print(f"  {candidate:>3}: {t:,} / {n:,}  ({100*t/n:.1f}%)")

Tokenizer loaded: microsoft/graphcodebert-base
Vocab size: 50,265
Model max context length: 512

Sampling token lengths (analysis pass: no padding, no truncation)...

Token length stats (4,085 functions):
  min    : 13
  mean   : 778.1
  median : 336
  p50    : 336
  p75    : 848
  p90    : 1989
  p95    : 3254
  p99    : 5850
  p100   : 8004

Truncated at max_length=512: 1,560 / 4,085  (38.2%)
  -> microsoft/graphcodebert-base is capped at 512 tokens. Use chunking/windowing or line-level localization to reduce information loss.

Estimated truncation by candidate max_length values:
  256: 2,414 / 4,085  (59.1%)
  384: 1,868 / 4,085  (45.7%)
  512: 1,560 / 4,085  (38.2%)


## 4 — Tokenize & Cache to Disk

In [22]:
import torch
from tqdm.auto import tqdm

CACHE_FILE = TOKEN_CACHE_DIR / f"tokens_maxlen{MAX_LENGTH}.pt"
CHUNK_STRIDE = int(cfg.get("chunk_stride", max(1, MAX_LENGTH // 2)))

# GraphCodeBERT has a hard 512-token context window.
MODEL_CTX = int(getattr(tokenizer, "model_max_length", MAX_LENGTH))
if MODEL_CTX is None or MODEL_CTX > 100000:
    MODEL_CTX = 512
if MAX_LENGTH > MODEL_CTX:
    print(f"Clamping MAX_LENGTH from {MAX_LENGTH} to model limit {MODEL_CTX}")
    MAX_LENGTH = MODEL_CTX

MAX_CONTENT_LEN = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)
assert MAX_CONTENT_LEN > 0, "Invalid max content length after special tokens"
if CHUNK_STRIDE > MAX_CONTENT_LEN:
    CHUNK_STRIDE = MAX_CONTENT_LEN

print(
    f"Chunk settings: max_length={MAX_LENGTH}, max_content_len={MAX_CONTENT_LEN}, stride={CHUNK_STRIDE}"
)

if CACHE_FILE.exists():
    print(f"Cache already exists: {CACHE_FILE}")
    print("Delete it to re-tokenize, or skip to next cell.")
else:
    print(f"Tokenizing {len(records):,} functions with chunked windowing...")

    all_input_ids = []
    all_attention_mask = []
    all_cwe_labels = []
    all_binary_labels = []
    all_global_ids = []
    all_split_origins = []
    all_chunk_index = []
    all_chunk_count = []

    total_chunks = 0
    functions_chunked = 0

    for r in tqdm(records, desc="Tokenizing (chunked)"):
        code = " ".join(r["lines"])
        content_ids = tokenizer.encode(
            code,
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )

        if len(content_ids) == 0:
            content_ids = [tokenizer.unk_token_id]

        starts = list(range(0, len(content_ids), CHUNK_STRIDE))
        if starts[-1] + MAX_CONTENT_LEN < len(content_ids):
            starts.append(len(content_ids) - MAX_CONTENT_LEN)
        starts = sorted(set(max(0, s) for s in starts))

        chunk_pieces = [content_ids[s : s + MAX_CONTENT_LEN] for s in starts]
        chunk_count = len(chunk_pieces)
        if chunk_count > 1:
            functions_chunked += 1

        for chunk_idx, piece in enumerate(chunk_pieces):
            text = tokenizer.decode(piece, skip_special_tokens=False)
            enc = tokenizer(
                text,
                add_special_tokens=True,
                max_length=MAX_LENGTH,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )

            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            all_cwe_labels.append(map_cwe_to_label(r["cwe_id"]))
            all_binary_labels.append(int(r["is_vulnerable"]))
            all_global_ids.append(r["global_id"])
            all_split_origins.append(r["split_origin"])
            all_chunk_index.append(chunk_idx)
            all_chunk_count.append(chunk_count)
            total_chunks += 1

        cache = {
            "input_ids": torch.stack([t.squeeze(0) for t in all_input_ids]),
            "attention_mask": torch.stack([t.squeeze(0) for t in all_attention_mask]),
            "cwe_labels": torch.tensor(all_cwe_labels, dtype=torch.long),
            "binary_labels": torch.tensor(all_binary_labels, dtype=torch.long),
            "global_ids": torch.tensor(all_global_ids, dtype=torch.long),
            "split_origins": all_split_origins,
            "chunk_index": torch.tensor(all_chunk_index, dtype=torch.long),
            "chunk_count": torch.tensor(all_chunk_count, dtype=torch.long),
            "max_length": MAX_LENGTH,
            "model_name": MODEL_NAME,
            "num_records": total_chunks,
            "num_functions": len(records),
            "chunk_stride": CHUNK_STRIDE,
        }

    torch.save(cache, CACHE_FILE)
    size_mb = CACHE_FILE.stat().st_size / 1024 / 1024
    print(f"\nSaved: {CACHE_FILE}  ({size_mb:.1f} MB)")
    print(f"  chunk records         : {cache['num_records']:,}")
    print(f"  original functions    : {cache['num_functions']:,}")
    print(f"  functions chunked     : {functions_chunked:,}")
    print(
        f"  avg chunks per function: {cache['num_records']/max(1, cache['num_functions']):.2f}"
    )
    print(f"  input_ids shape       : {cache['input_ids'].shape}")
    print(f"  attention_mask shape  : {cache['attention_mask'].shape}")
    print(f"  cwe_labels shape      : {cache['cwe_labels'].shape}")
    print(f"  binary_labels shape   : {cache['binary_labels'].shape}")

Chunk settings: max_length=512, max_content_len=510, stride=256
Tokenizing 4,085 functions with chunked windowing...


Tokenizing (chunked): 100%|██████████| 4085/4085 [02:05<00:00, 32.51it/s]



Saved: /Users/anas/Projects/code-security-identifier/datasets/token_cache/tokens_maxlen512.pt  (114.1 MB)
  chunk records         : 14,522
  original functions    : 4,085
  functions chunked     : 2,407
  avg chunks per function: 3.55
  input_ids shape       : torch.Size([14522, 512])
  attention_mask shape  : torch.Size([14522, 512])
  cwe_labels shape      : torch.Size([14522])
  binary_labels shape   : torch.Size([14522])


## 5 — Verify Cache Integrity

In [23]:
cache = torch.load(CACHE_FILE, weights_only=True)
N = cache["num_records"]

assert cache["input_ids"].shape == (N, MAX_LENGTH), "input_ids shape mismatch"
assert cache["attention_mask"].shape == (N, MAX_LENGTH), "attention_mask shape mismatch"
assert cache["cwe_labels"].shape == (N,), "cwe_labels shape mismatch"
assert cache["binary_labels"].shape == (N,), "binary_labels shape mismatch"
assert len(cache["split_origins"]) == N, "split_origins length mismatch"
assert cache["max_length"] == MAX_LENGTH
assert cache["model_name"] == MODEL_NAME

# Label sanity
assert cache["cwe_labels"].min() >= 0 and cache["cwe_labels"].max() <= 7
assert cache["binary_labels"].min() >= 0 and cache["binary_labels"].max() <= 1

train_mask = [s == "train" for s in cache["split_origins"]]
val_mask = [s == "val" for s in cache["split_origins"]]

num_functions = int(cache.get("num_functions", len(set(cache["global_ids"].tolist()))))
chunk_count = cache.get("chunk_count")
if isinstance(chunk_count, torch.Tensor):
    avg_chunks = float(chunk_count.float().mean().item())
else:
    avg_chunks = N / max(1, num_functions)

print(f"Cache VERIFIED: {N:,} chunk records")
print(f"  functions       : {num_functions:,}")
print(f"  avg chunks/func : {avg_chunks:.2f}")
print(
    f'  input_ids       : {tuple(cache["input_ids"].shape)}  dtype={cache["input_ids"].dtype}'
)
print(f'  attention_mask  : {tuple(cache["attention_mask"].shape)}')
print(
    f'  cwe_labels      : min={cache["cwe_labels"].min().item()}  max={cache["cwe_labels"].max().item()}'
)
print(
    f'  binary_labels   : {cache["binary_labels"].sum().item():,} vulnerable / {N:,} total chunks'
)
print(f"  train split     : {sum(train_mask):,} chunks")
print(f"  val split       : {sum(val_mask):,} chunks")

Cache VERIFIED: 14,522 chunk records
  functions       : 4,085
  avg chunks/func : 9.24
  input_ids       : (14522, 512)  dtype=torch.int64
  attention_mask  : (14522, 512)
  cwe_labels      : min=0  max=7
  binary_labels   : 11,321 vulnerable / 14,522 total chunks
  train split     : 12,994 chunks
  val split       : 1,528 chunks


## 6 — VulnerabilityDataset + get_dataloaders()

In [24]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np


class VulnerabilityDataset(Dataset):
    """
    Loads pre-tokenized chunk tensors from cache.
    Returns dict with input_ids, attention_mask, cwe_label, binary_label, global_id.
    """

    def __init__(self, cache: dict, split: str):
        """
        split: 'train' | 'val' | 'all'
        """
        if split == "all":
            indices = list(range(len(cache["split_origins"])))
        else:
            indices = [i for i, s in enumerate(cache["split_origins"]) if s == split]

        self.input_ids = cache["input_ids"][indices]
        self.attention_mask = cache["attention_mask"][indices]
        self.cwe_labels = cache["cwe_labels"][indices]
        self.binary_labels = cache["binary_labels"][indices]
        self.global_ids = cache["global_ids"][indices]
        self.chunk_index = cache.get("chunk_index")
        self.chunk_count = cache.get("chunk_count")
        if isinstance(self.chunk_index, torch.Tensor):
            self.chunk_index = self.chunk_index[indices]
        if isinstance(self.chunk_count, torch.Tensor):
            self.chunk_count = self.chunk_count[indices]

        self.split = split

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "cwe_label": self.cwe_labels[idx],
            "binary_label": self.binary_labels[idx],
            "global_id": self.global_ids[idx],
        }
        if isinstance(self.chunk_index, torch.Tensor):
            item["chunk_index"] = self.chunk_index[idx]
        if isinstance(self.chunk_count, torch.Tensor):
            item["chunk_count"] = self.chunk_count[idx]
        return item


def make_balanced_sampler(dataset: VulnerabilityDataset) -> WeightedRandomSampler:
    """
    Weighted sampler that upsamples minority CWE classes.
    Weight per sample = 1 / class_count.
    """
    labels = dataset.cwe_labels.numpy()
    class_counts = np.bincount(labels, minlength=8)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float),
        num_samples=len(labels),
        replacement=True,
    )


def get_dataloaders(
    cache: dict, batch_size: int = 16, balanced: bool = True, num_workers: int = 0
):
    """
    Returns (train_loader, val_loader).
    balanced=True applies WeightedRandomSampler on train set.
    num_workers=0 recommended for Colab / MPS to avoid fork issues.
    """
    train_ds = VulnerabilityDataset(cache, split="train")
    val_ds = VulnerabilityDataset(cache, split="val")

    if balanced:
        sampler = make_balanced_sampler(train_ds)
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=num_workers,
            pin_memory=True,
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True,
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader


print("VulnerabilityDataset + get_dataloaders() defined (chunk-aware)")

VulnerabilityDataset + get_dataloaders() defined (chunk-aware)


## 7 — Smoke Test DataLoaders

In [26]:
train_loader, val_loader = get_dataloaders(
    cache, batch_size=cfg["batch_size"], balanced=True
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")

num_functions = int(cache.get("num_functions", len(set(cache["global_ids"].tolist()))))
num_chunks = int(cache["num_records"])
print(f"Functions     : {num_functions:,}")
print(f"Chunks        : {num_chunks:,}")
print(f"Avg chunks/fn : {num_chunks / max(1, num_functions):.2f}")

# One batch
batch = next(iter(train_loader))
print("\nSample batch:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<16} shape={tuple(v.shape)}  dtype={v.dtype}")

# CWE label distribution in this batch
cwe_in_batch = Counter(batch["cwe_label"].tolist())
print(f"\nCWE labels in batch: {dict(sorted(cwe_in_batch.items()))}")

print("\nDataLoader smoke test PASSED")
print("\nReady for 02_training.ipynb")
print(f"Token cache: {CACHE_FILE}")

Train batches : 813
Val batches   : 96
Functions     : 4,085
Chunks        : 14,522
Avg chunks/fn : 3.55

Sample batch:
  input_ids        shape=(16, 512)  dtype=torch.int64
  attention_mask   shape=(16, 512)  dtype=torch.int64
  cwe_label        shape=(16,)  dtype=torch.int64
  binary_label     shape=(16,)  dtype=torch.int64
  global_id        shape=(16,)  dtype=torch.int64
  chunk_index      shape=(16,)  dtype=torch.int64
  chunk_count      shape=(16,)  dtype=torch.int64

CWE labels in batch: {0: 3, 1: 1, 2: 1, 3: 2, 4: 1, 5: 2, 6: 5, 7: 1}

DataLoader smoke test PASSED

Ready for 02_training.ipynb
Token cache: /Users/anas/Projects/code-security-identifier/datasets/token_cache/tokens_maxlen512.pt
